# Configuration du projet

### Clônage du dépôt

In [ ]:
!git clone https://github.com/wydad-fulbert/NegativePrompt.git
%cd NegativePrompt


Cloning into 'NegativePrompt'...
remote: Enumerating objects: 446, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 446 (delta 53), reused 60 (delta 21), pack-reused 353 (from 1)
Receiving objects: 100% (446/446), 4.91 MiB | 12.80 MiB/s, done.
Resolving deltas: 100% (212/212), done.
/kaggle/working/NegativePrompt/NegativePrompt/NegativePrompt


### Installation des dépendances

In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.7 MB/s eta 0:00:00


In [ ]:
!pip install -U transformers accelerate bitsandbytes sentencepiece
!pip install fire
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import pandas as pd

### Vérification du GPU

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


# **PARTIE I — Reproduction solide des baselines**

### PROTOCOLE

**1. Modèles**

 * Flan-T5-Large

 * Llama-2-7B-Chat

 * Vicuna-7B-v1.5

**2. Tâches**


* *Instruction Induction (5)* :
   - sentiment

   - translation_en-fr

   - word_in_context

   - active_to_passive

   - negation


* *BigBench (5)* :

  - dyck_languages

  - object_counting

  - ruin_names

  - word_sorting

  - disambiguation_qa

**3. Stimuli**




* 0 (baseline)
  ,1, 5, et 10


Stimulus 0 correspond à la baseline (sans Negative Prompt).
Stimulus 1, 5 et 10 correspondent respectivement aux prompts NP01, NP05 et NP10.





**4. Paramètres**

  * Zero-shot (few_shot=False)

  * Température = 0

  * max_new_tokens = 50

  * num_samples = 100

  * seed = 42



### Modèle Flan-T5-Large

In [ ]:
!python run_t5.py

In [ ]:
import pandas as pd
dt5 = pd.read_csv("results_t5.csv")
dt5.head(10)



### Modèle Llama-2-7B-Chat

In [ ]:
### Accès à HuggingFace
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
!python run_llama2.py

In [ ]:
dllama = pd.read_csv("/content/NegativePrompt/NegativePrompt/RESULTS/results_10_tasks/results_llama2.csv")
dllama.head(10)


### Modèle Vicuna-7B-v1.5

In [ ]:
! python run_vicuna.py

In [ ]:
import pandas as pd
dvicuna = pd.read_csv("/content/NegativePrompt/NegativePrompt/RESULTS/results_10_tasks/results_vicuna.csv")
dvicuna.head(5)


## **Analyse des résultats**

In [ ]:
# Fusion des 3 fichiers résultats
import pandas as pd
t5 = pd.read_csv("/content/NegativePrompt/NegativePrompt/RESULTS/results_10_tasks/results_t5.csv")
llama = pd.read_csv("/content/NegativePrompt/NegativePrompt/RESULTS/results_10_tasks/results_llama2.csv")
vicuna = pd.read_csv("/content/NegativePrompt/NegativePrompt/RESULTS/results_10_tasks/results_vicuna.csv")
df = pd.concat([t5, llama, vicuna])


### Performance moyenne globale par modèle

In [ ]:
df.groupby("model")["score"].mean()

La figure présente la performance moyenne des trois modèles sur l’ensemble des tâches.

On observe que **Llama2** obtient les meilleurs résultats globaux (≈0.35), suivi de **Vicuna** (≈0.26), tandis que **Flan-T5** présente les performances les plus faibles (≈0.22).

Cependant, cette moyenne globale peut masquer des différences importantes selon les tâches, ce qui motive l’analyse détaillée présentée dans la section suivante.

In [ ]:
sns.barplot(
    data=df,
    x="model",
    y="score"
)

plt.title("Performance moyenne par modèle")
plt.show()

### Analyse par tâche

In [ ]:
df.groupby(["task","model"])["score"].mean().round(2).unstack()

In [ ]:
sns.barplot(
    data=df,
    x="task",
    y="score",
    hue="model"
)

plt.xticks(rotation=45)
plt.title("Performance par tâche et par modèle")
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

sns.barplot(
    data=df,
    x="task",
    y="score",
    hue="model",
    errorbar=None
)

plt.xticks(rotation=30, ha="right")
plt.title("Performance par tâche et par modèle")
plt.ylabel("Score")
plt.xlabel("Tâche")

plt.tight_layout()
plt.show()

L'analyse par tâche révèle des variations importantes entre les modèles.

Par exemple :

- **active_to_passive** est presque parfaitement résolue par Llama2 (**0.97**) alors que T5 échoue presque complètement (**0.02**).
- **sentiment** est mieux traité par T5 (**0.93**) que par les autres modèles.
- **object_counting** est légèrement mieux résolu par Vicuna (**0.26**).

Certaines tâches comme **ruin_names** ou **word_sorting** restent très difficiles pour tous les modèles (scores proches de 0).

### Effets des Negative Prompts

In [ ]:
df.groupby(["stimulus","model"])["score"].mean().round(2).unstack()

In [ ]:
sns.barplot(
    data=df,
    x="stimulus",
    y="score",
    hue="model"
)

On observe que les performances de **T5 restent quasiment constantes** (≈0.21–0.22) quel que soit le stimulus.

Cela suggère que les Negative Prompts ont **peu d'effet sur ce modèle**, contrairement à Llama2 et Vicuna qui montrent des variations plus importantes selon le stimulus.

Ce résultat confirme l'observation mentionnée dans le papier : certains modèles semblent plus sensibles aux modifications du prompt que d'autres.

## **Limites de la métrique Exact Match (EM) et métrique alternative**

La métrique Exact Match (EM) est très stricte pour certaines tâches génératives, comme la traduction. Deux réponses proches peuvent être considérées comme incorrectes si elles ne sont pas exactement identiques.

Pour illustrer cette limite, nous calculons également une mesure de similarité textuelle : le score **BLEU**, qui compare la réponse générée par le modèle à la réponse de référence.

In [ ]:
# Script sur les 3 modèles , 3 tâches , incluant  la métrique BLEU
!python run_3models_bleu.py

Afin d’illustrer concrètement le comportement des modèles, nous affichons quelques exemples de prédictions pour trois tâches génératives.

Pour chaque tâche et chaque modèle, nous montrons un exemple avec la condition **baseline (stimulus = 0)** et un exemple avec **Negative Prompt (stimulus = 5)**.

Le tableau présente l’entrée (input), la réponse attendue (reference), la prédiction du modèle (prediction), ainsi que les scores Exact Match et BLEU.

In [ ]:
import pandas as pd
bleu_df = pd.read_csv("/content/NegativePrompt/RESULTS/results_10_tasks/results_bleu_3models_detailed.csv")
tasks_to_show = ["translation_en-fr", "active_to_passive", "ruin_names"]
stimuli_to_show = [0, 5]

examples = (
    bleu_df[
        (bleu_df["task"].isin(tasks_to_show)) &
        (bleu_df["stimulus"].isin(stimuli_to_show))
    ]
    .groupby(["model","task","stimulus"])
    .head(1)
    [["model","task","stimulus","input","reference","prediction","em_normalized","bleu"]]
)
examples = examples.sort_values(["task","model","stimulus"])
examples

In [ ]:
bleu_df.groupby(["model","task","stimulus"])[["em_normalized","bleu"]].mean().round(2)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

bleu_summary = bleu_df.groupby(["model","task"])[["em_normalized","bleu"]].mean().reset_index()

plt.figure(figsize=(8,4))

sns.barplot(
    data=bleu_summary,
    x="task",
    y="bleu",
    hue="model"
)

plt.title("Score BLEU par tâche et par modèle")
plt.xticks(rotation=30)
plt.show()

#### Comparaison entre Exact Match (EM) et BLEU

Les résultats montrent que les scores BLEU sont généralement plus élevés que les scores Exact Match pour certaines tâches génératives.

Par exemple, pour la tâche **active_to_passive**, les modèles obtiennent des scores BLEU élevés même lorsque le score Exact Match est plus faible. Cela signifie que les réponses générées sont souvent proches de la référence, même si elles ne sont pas identiques.

Pour la tâche **translation_en-fr**, on observe également que les scores BLEU capturent une certaine similarité entre la prédiction et la référence, alors que Exact Match reste souvent nul.

Ces résultats illustrent la limite de la métrique Exact Match : elle peut sous-estimer la qualité des réponses générées lorsque plusieurs formulations correctes sont possibles.

# **PARTIE II — Ablations : contrôles de longueur et de saillance**

Nous comparons quatre conditions :

- **baseline** : prompt original sans Negative Prompt  
- **np01** : ajout du Negative Prompt NP01  
- **neutral_salience** : ajout d’un marqueur saillant mais neutre ("IMPORTANT:")  
- **neutral_length** : ajout d’un texte neutre de longueur comparable  

Ces contrôles permettent de vérifier si l’effet observé avec les Negative Prompts provient réellement de leur contenu, ou plus généralement du fait d’ajouter un texte supplémentaire au prompt.

Nous analysons ensuite si les performances obtenues avec les contrôles neutres sont proches de celles obtenues avec NP01.

In [ ]:
### Accès à HuggingFace
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

Les expériences d’ablation sont en cours d’exécution au moment de l’envoi de ce notebook. Les résultats présentés ici concernent principalement les modèles T5 et Llama2, tandis que l’exécution pour Vicuna n’est pas encore terminée en raison des contraintes de calcul. L’analyse ci-dessous doit donc être interprétée comme une analyse préliminaire.

In [ ]:
#expériences d’ablation sur 3 modèles et 5 tâches, en comparant : baseline,Negative Prompt NP01, deux contrôles neutres de longueur et de saillance.
!python run_control.py

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

controls = pd.read_csv("results_controls_all_models.csv")
controls.head()

Pour le modèle T5, les résultats montrent des écarts très faibles entre les différentes conditions. Les différences entre baseline, NP01 et les contrôles neutres restent généralement inférieures à 0.03 selon les tâches.

Cela suggère que T5 est globalement peu sensible aux modifications du prompt dans notre protocole expérimental, ce qui confirme également l’observation faite dans l’analyse précédente sur la stabilité de ce modèle face aux Negative Prompts.

In [ ]:
controls.groupby(["model", "task", "condition"]).size().reset_index(name="n")

Pour Llama2, les résultats sont plus variables selon les tâches. Dans certains cas, NP01 apporte une légère amélioration (par exemple +0.02 sur translation_en-fr), tandis que dans d’autres cas les contrôles neutres produisent des effets comparables voire supérieurs (par exemple +0.08 pour neutral_salience sur object_counting).

Ces résultats suggèrent que l’effet du Negative Prompt n’est pas systématiquement lié à son contenu, et qu’une partie de l’effet pourrait être expliquée par des facteurs plus généraux comme la saillance ou la longueur du texte ajouté.

In [ ]:
pivot_diff = controls.pivot_table(
    index=["model", "task"],
    columns="condition",
    values="score"
)

pivot_diff["np01_minus_baseline"] = pivot_diff["np01"] - pivot_diff["baseline"]
pivot_diff["neutral_length_minus_baseline"] = pivot_diff["neutral_length"] - pivot_diff["baseline"]
pivot_diff["neutral_salience_minus_baseline"] = pivot_diff["neutral_salience"] - pivot_diff["baseline"]

pivot_diff[[
    "np01_minus_baseline",
    "neutral_length_minus_baseline",
    "neutral_salience_minus_baseline"
]].round(3)

In [ ]:
diff_summary = pivot_diff[[
    "np01_minus_baseline",
    "neutral_length_minus_baseline",
    "neutral_salience_minus_baseline"
]].mean().round(3)

diff_summary

L’objectif de ces expériences est de déterminer si l’effet observé avec les Negative Prompts provient réellement de leur contenu, ou simplement du fait d’ajouter du texte supplémentaire au prompt.

Pour cela, nous comparons quatre conditions : baseline, NP01, neutral_salience (marqueur neutre) et neutral_length (texte neutre de longueur comparable).

Dans l’ensemble, les résultats préliminaires indiquent que les Negative Prompts n’apportent pas d’amélioration systématique des performances. Les écarts observés restent faibles et parfois comparables à ceux obtenus avec des contrôles neutres.

Ces observations vont dans le sens d’une interprétation prudente de l’effet des Negative Prompts, qui pourrait en partie s’expliquer par des facteurs liés à la structure du prompt plutôt qu’au contenu sémantique lui-même.

Une analyse complète sera réalisée une fois l’ensemble des modèles exécuté.

# **Partie III : Améliorations**

### Test 1 NP auto generated

In [ ]:
!python run_np_auto.py

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

npauto = pd.read_csv("results_np_auto.csv")
npauto.head()

In [ ]:
npauto.head()

In [ ]:
npauto.groupby(["model","condition"])["score"].mean().unstack().round(3)

In [ ]:
npauto.pivot_table(
    index=["model","task"],
    columns="condition",
    values="score"
).round(2)

In [ ]:
!python run_np_auto_llama2.py

### Test finetuning LoRa

In [ ]:
!python build_lora_dataset.py

In [ ]:
###UNIQUEMENT SUR KAGGLE (ACCES A HUGGING FACE)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)
print("HF OK")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF OK


In [ ]:
!pip install -q transformers datasets peft accelerate sentencepiece evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.0 MB/s eta 0:00:00


In [ ]:
!python train_t5_lora.py

Generating train split: 56688 examples [00:00, 402523.67 examples/s]
config.json: 100%|█████████████████████████████| 662/662 [00:00<00:00, 2.00MB/s]
tokenizer_config.json: 2.54kB [00:00, 930kB/s]
spiece.model: 100%|██████████████████████████| 792k/792k [00:00<00:00, 2.52MB/s]
tokenizer.json: 2.42MB [00:00, 19.3MB/s]
special_tokens_map.json: 2.20kB [00:00, 6.09MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|████████████████████| 3.13G/3.13G [00:07<00:00, 433MB/s]
Loading weights: 100%|███████████████████████| 558/558 [00:00<00:00, 796.90it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
generation_config.json: 100%|███████████████████| 147/147 [00:00<00:00, 709kB/s]
trainable params: 2,359,296 || all params: 785,509,376 || trainable%: 0.3004
  0%|   

##### Vérification du modèle (absence de vide, d'absurdité...)

In [ ]:
!python debug_t5_lora.py

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 558/558 [00:00<00:00, 910.22it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|███████████████████████| 558/558 [00:00<00:00, 939.24it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning

PROMPT: Instruction: Determine whether a movie review is positive or negative.

Input: This film was excellent and moving.
Answer:
BASE: positive
LORA: 

PROMPT: Instruction: Translate the word into French.

Input: hello
Answer:
BASE: Hello
LORA: 

PROMPT: Instruction: Count the n

##### Evaluation

In [ ]:
!python eval_t5_lora.py

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 558/558 [00:00<00:00, 948.41it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|███████████████████████| 558/558 [00:00<00:00, 963.68it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning

=== CONDITION: baseline ===

=== CONDITION: np01 ===

=== CONDITION: np05 ===

=== CONDITION: np10 ===
                 task condition    metric_used  score_base  score_lora
0   active_to_passive  baseline           bleu    0.143427    0.000000
1   active_to_passive      np01     

In [ ]:
from google.colab import files
files.upload()

Saving results_t5_lora_eval_fixed.csv to results_t5_lora_eval_fixed.csv


{'results_t5_lora_eval_fixed.csv': b'task,condition,metric_used,score_base,score_lora\nactive_to_passive,baseline,bleu,0.1434273302541761,0.0\nactive_to_passive,np01,bleu,0.11389034164225112,0.0\nactive_to_passive,np05,bleu,0.11311311875408814,0.0\nactive_to_passive,np10,bleu,0.11165325332949196,0.0\ndisambiguation_qa,baseline,normalized_em,0.0,0.0\ndisambiguation_qa,np01,normalized_em,0.0,0.0\ndisambiguation_qa,np05,normalized_em,0.0,0.0\ndisambiguation_qa,np10,normalized_em,0.0,0.0\ndyck_languages,baseline,normalized_em,0.5333333333333333,0.26666666666666666\ndyck_languages,np01,normalized_em,0.4666666666666667,0.26666666666666666\ndyck_languages,np05,normalized_em,0.5666666666666667,0.26666666666666666\ndyck_languages,np10,normalized_em,0.5,0.26666666666666666\nnegation,baseline,normalized_em,0.0,0.0\nnegation,np01,normalized_em,0.0,0.0\nnegation,np05,normalized_em,0.0,0.0\nnegation,np10,normalized_em,0.0,0.0\nobject_counting,baseline,normalized_em,0.26666666666666666,0.0\nobject_co

In [ ]:
import pandas as pd
df_lora = pd.read_csv("results_t5_lora_eval_fixed.csv")
df_lora["gain"] = df_lora["score_lora"] - df_lora["score_base"]
df_lora[["task","condition","metric_used","score_base","score_lora","gain"]].round(4)

,task,condition,metric_used,score_base,score_lora,gain
0,active_to_passive,baseline,bleu,0.1434,0.0000,-0.1434
1,active_to_passive,np01,bleu,0.1139,0.0000,-0.1139
2,active_to_passive,np05,bleu,0.1131,0.0000,-0.1131
3,active_to_passive,np10,bleu,0.1117,0.0000,-0.1117
4,disambiguation_qa,baseline,normalized_em,0.0000,0.0000,0.0000
5,disambiguation_qa,np01,normalized_em,0.0000,0.0000,0.0000
6,disambiguation_qa,np05,normalized_em,0.0000,0.0000,0.0000
7,disambiguation_qa,np10,normalized_em,0.0000,0.0000,0.0000
8,dyck_languages,baseline,normalized_em,0.5333,0.2667,-0.2667
9,dyck_languages,np01,normalized_em,0.4667,0.2667,-0.2000


### Test n°2 amélioaration NP auto generated

In [ ]:
## Installation des dépendances
! pip install transformers datasets nltk -q

##### Test sur Flan-T5-Large

In [ ]:
!python eval_final_np_t5.py

config.json: 100%|█████████████████████████████| 662/662 [00:00<00:00, 2.15MB/s]
tokenizer_config.json: 2.54kB [00:00, 6.45MB/s]
spiece.model: 100%|██████████████████████████| 792k/792k [00:00<00:00, 1.76MB/s]
tokenizer.json: 2.42MB [00:00, 54.9MB/s]
special_tokens_map.json: 2.20kB [00:00, 6.73MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|████████████████████| 3.13G/3.13G [00:07<00:00, 429MB/s]
Loading weights: 100%|█| 558/558 [00:00<00:00, 609.49it/s, Materializing param=s
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
generation_config.json: 100%|███████████████████| 147/147 [00:00<00:00, 527kB/s]

=== baseline ===

=== np01 ===

=== np05 ===

=== np10 ===

=== np_generated ===
                 task     condition metric     score
0   active_to_pass

In [ ]:
import os

filename = "results_final_np.csv"  # Remplacez par le nom exact de votre fichier
for root, dirs, files in os.walk('/content'):
    if filename in files:
        print(os.path.join(root, filename))

/content/NegativePrompt/NegativePrompt/NegativePrompt/results_final_np.csv
/content/NegativePrompt/NegativePrompt/NegativePrompt/NegativePrompt/results_final_np.csv


In [ ]:
import pandas as pd

# Ajoutez des guillemets autour du chemin
df = pd.read_csv("/kaggle/working/NegativePrompt/NegativePrompt/results_final_np_t5.csv")

# Pour vérifier que le chargement a fonctionné
df.head(20)

,task,condition,metric,score
0,active_to_passive,baseline,bleu,0.143427
1,active_to_passive,np01,bleu,0.113890
2,active_to_passive,np05,bleu,0.113113
3,active_to_passive,np10,bleu,0.111653
4,active_to_passive,np_generated,bleu,0.172964
5,disambiguation_qa,baseline,em,0.000000
6,disambiguation_qa,np01,em,0.000000
7,disambiguation_qa,np05,em,0.000000
8,disambiguation_qa,np10,em,0.000000
9,disambiguation_qa,np_generated,em,0.066667


##### Test sur Llama-2-7B-Chat

In [ ]:
#### UNIQUEMENT POUR COLAB
### Accès à HuggingFace
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
!python eval_final_np_llama2.py

config.json: 100%|█████████████████████████████| 614/614 [00:00<00:00, 2.65MB/s]
tokenizer_config.json: 1.62kB [00:00, 4.32MB/s]
tokenizer.json: 1.84MB [00:00, 41.6MB/s]
tokenizer.model: 100%|███████████████████████| 500k/500k [00:00<00:00, 2.04MB/s]
special_tokens_map.json: 100%|█████████████████| 414/414 [00:00<00:00, 2.41MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 26.8kB [00:00, 60.1MB/s]
Fetching 2 files: 100%|███████████████████████████| 2/2 [00:39<00:00, 19.62s/it]
Download complete: 100%|████████████████████| 13.5G/13.5G [00:39<00:00, 342MB/s]
Loading weights: 100%|█| 291/291 [00:03<00:00, 78.03it/s, Materializing param=mo
generation_config.json: 100%|██████████████████| 188/188 [00:00<00:00, 1.13MB/s]

=== baseline ===
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

=== np01 ===

=== np05 ===

=== np10 ===

=== np_generated ===
              

In [ ]:
import pandas as pd

df_llama2 = pd.read_csv("/kaggle/working/NegativePrompt/results_final_np_llama2.csv")
df_llama2.round(3)

,task,condition,metric,score
0,active_to_passive,baseline,bleu,0.260
1,active_to_passive,np01,bleu,0.182
2,active_to_passive,np05,bleu,0.183
3,active_to_passive,np10,bleu,0.157
4,active_to_passive,np_generated,bleu,0.152
5,disambiguation_qa,baseline,em,0.000
6,disambiguation_qa,np01,em,0.000
7,disambiguation_qa,np05,em,0.000
8,disambiguation_qa,np10,em,0.000
9,disambiguation_qa,np_generated,em,0.000


##### Test sur Vicuna-7B-v1.5

In [ ]:
!python eval_final_np_vicuna.py

tokenizer_config.json: 100%|███████████████████| 749/749 [00:00<00:00, 3.42MB/s]
tokenizer.model: 100%|███████████████████████| 500k/500k [00:00<00:00, 2.03MB/s]
special_tokens_map.json: 100%|█████████████████| 438/438 [00:00<00:00, 2.22MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin.index.json: 26.8kB [00:00, 15.0MB/s]

model.safetensors.index.json: 28.1kB [00:00, 37.2MB/s]A
Fetching 2 files: 100%|███████████████████████████| 2/2 [00:41<00:00, 20.75s/it]
Download complete: 100%|████████████████████| 13.5G/13.5G [00:41<00:00, 325MB/s]
Loading weights: 100%|█| 291/291 [00:04<00:00, 71.04it/s, Materializing param=mo
generation_config.json: 100%|███████████████████| 162/162 [00:00<00:00, 958kB/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

=== baseline ===

=== np01 ===

=== np05 ===

=== np10 ===

=== np_generated ===
                 task     condition metric    

In [ ]:
import pandas as pd

df_vic = pd.read_csv("/kaggle/working/NegativePrompt/NegativePrompt/results_final_np_vicuna.csv")
df_vic.round(3)

,task,condition,metric,score
0,active_to_passive,baseline,bleu,0.008
1,active_to_passive,np01,bleu,0.005
2,active_to_passive,np05,bleu,0.006
3,active_to_passive,np10,bleu,0.005
4,active_to_passive,np_generated,bleu,0.005
5,disambiguation_qa,baseline,em,0.000
6,disambiguation_qa,np01,em,0.000
7,disambiguation_qa,np05,em,0.000
8,disambiguation_qa,np10,em,0.000
9,disambiguation_qa,np_generated,em,0.000


In [ ]:
!python eval_np_mistral.py

config.json: 100%|█████████████████████████████| 571/571 [00:00<00:00, 2.16MB/s]
tokenizer_config.json: 2.10kB [00:00, 4.71MB/s]
tokenizer.json: 1.80MB [00:00, 13.8MB/s]
tokenizer.model: 100%|████████████████████████| 493k/493k [00:00<00:00, 968kB/s]
special_tokens_map.json: 100%|█████████████████| 414/414 [00:00<00:00, 3.08MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 25.1kB [00:00, 47.7MB/s]
Fetching 2 files: 100%|███████████████████████████| 2/2 [00:48<00:00, 24.26s/it]
Download complete: 100%|████████████████████| 14.5G/14.5G [00:48<00:00, 298MB/s]
Loading weights: 100%|█| 291/291 [00:14<00:00, 20.10it/s, Materializing param=mo
generation_config.json: 100%|███████████████████| 116/116 [00:00<00:00, 377kB/s]

=== baseline ===

=== np01 ===

=== np05 ===

=== np10 ===
                 task condition metric     score
0   active_to_passive  baseline   bleu  1.000000
1   active_to_passive      np01   bleu  0.967995
2   active_to_passive      np05 